# Phase 4 - Engine failure-mode head (NEW in v3)  —  Rotax_916_iS

Misfire, injector fouling, cooling degradation, combustion instability - PS section C.

These are **engine** faults, not sensor faults. v2 had no such head: those words
existed only in `advisory.py` as labels mapped onto sensor channels.

In [ ]:
import os, sys, json, time
sys.path.insert(0, os.path.abspath("."))
import numpy as np, tensorflow as tf
from tensorflow import keras
import tf_data_pipeline as P, model_architectures as A, train_common as C

C.report_gpu()

In [ ]:
# ---- SET THE ENGINE HERE. Run this notebook once per engine. ----
ENGINE = "Rotax_916_iS"
steps = C.describe(ENGINE)
train_ds, val_ds, test_ds = C.datasets(ENGINE)

### Why sigmoid, not softmax
Several modes can be present at once, so these are independent severities rather
than a one-of-N choice. That is the structural difference from the diagnosis head.

In [ ]:
xi  = A.build_encoder_input()
enc = A.build_encoder(xi)
fm  = A.build_failure_mode_head(enc)
model = keras.Model(xi, {"y_failure_mode": fm}, name="failure_modes")
model.summary()

In [ ]:
# (nothing extra for this phase)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss={"y_failure_mode": keras.losses.BinaryCrossentropy()},
              metrics={"y_failure_mode": [keras.metrics.MeanAbsoluteError(name="mae")]
                                           + C.failure_mode_metrics()})

In [ ]:
PHASE = "phase4_failure_modes"
PREV  = "phase3_severity"
ckpt  = C.ckpt_path(ENGINE, PHASE)
if PREV: C.warm_start(model, C.ckpt_path(ENGINE, PREV))

hist = model.fit(
    train_ds.map(C.split_labels(["y_failure_mode"])),
    validation_data=val_ds.map(C.split_labels(["y_failure_mode"])),
    steps_per_epoch=steps["train"],
    validation_steps=steps["val"],
    epochs=20,
    callbacks=C.callbacks(ckpt, monitor="val_auc_all_det", mode="max"),
    verbose=1,
)
print("saved:", ckpt)

In [ ]:
res = model.evaluate(test_ds.map(C.split_labels(["y_failure_mode"])), steps=steps["test"], verbose=1, return_dict=True)
print(json.dumps({k: float(v) for k, v in res.items()}, indent=2))
with open(os.path.join(C.MODELS_DIR, f"{PHASE}_{C.ENGINE_KEY[ENGINE]}_test.json"), "w") as f:
    json.dump({k: float(v) for k, v in res.items()}, f, indent=2)